**Deep Learning Experiment Tracking and Model Management Using MLflow**

In [1]:

%pip install -q mlflow scikit-learn pandas matplotlib


In [2]:

import tensorflow as tf
import mlflow
import mlflow.tensorflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)
print("Libraries imported successfully!")


TensorFlow version: 2.21.0
GPU available: False
Libraries imported successfully!


In [3]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Dataset loaded successfully!")

print("\nTraining data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

print("\nTraining labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("\nNumber of classes:", len(np.unique(y_train)))


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset loaded successfully!

Training data shape: (60000, 28, 28)
Testing data shape: (10000, 28, 28)

Training labels: (60000,)
Testing labels: (10000,)

Number of classes: 10


In [4]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Preprocessing completed!")

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())


Preprocessing completed!
Training data shape: (60000, 28, 28, 1)
Testing data shape: (10000, 28, 28, 1)
Minimum pixel value: 0.0
Maximum pixel value: 1.0


In [5]:

def create_cnn_model():

    model = models.Sequential([

        # Define input shape
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(64, activation='relu'),

        layers.Dense(10, activation='softmax')
    ])

    return model


model = create_cnn_model()
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:

mlflow.set_experiment("CNN_Experiment_Tracking")

mlflow.set_tracking_uri("sqlite:///mlflow.db")

experiment = mlflow.set_experiment(
    "CNN_Experiment_Tracking"
)

print("MLflow configured successfully!")
print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)


2026/11/19 01:23:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/19 01:23:50 INFO mlflow.store.db.utils: Updating database tables
2026/11/19 01:23:54 INFO mlflow.tracking.fluent: Experiment with name 'CNN_Experiment_Tracking' does not exist. Creating a new experiment.


MLflow configured successfully!
Experiment ID: 1
Experiment Name: CNN_Experiment_Tracking


In [7]:

def run_experiment(learning_rate, epochs, run_name):

    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", 128)
        mlflow.log_param("optimizer", "Adam")
        mlflow.log_param("architecture", "CNN")

        model = create_cnn_model()

        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        history = model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=128,
            validation_split=0.1,
            verbose=1
        )

        test_loss, test_accuracy = model.evaluate(
            x_test, y_test, verbose=0
        )

        mlflow.log_metric("test_loss", float(test_loss))
        mlflow.log_metric("test_accuracy", float(test_accuracy))
        mlflow.log_metric(
            "final_train_accuracy",
            float(history.history["accuracy"][-1])
        )
        mlflow.log_metric(
            "final_val_accuracy",
            float(history.history["val_accuracy"][-1])
        )

        mlflow.tensorflow.log_model(
            model,
            name="cnn_model"
        )

        print(f"\nExperiment: {run_name}")
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Loss: {test_loss:.4f}")

    return model, history


In [8]:

# Run Experiment 1
model1, history1 = run_experiment(
    learning_rate=0.001,
    epochs=3,
    run_name="CNN_Experiment_1"
)

# Run Experiment 2
model2, history2 = run_experiment(
    learning_rate=0.0005,
    epochs=3,
    run_name="CNN_Experiment_2"
)

# Run Experiment 3
model3, history3 = run_experiment(
    learning_rate=0.001,
    epochs=5,
    run_name="CNN_Experiment_3"
)

print("\nAll three experiments completed!")


Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - accuracy: 0.9304 - loss: 0.2408 - val_accuracy: 0.9768 - val_loss: 0.0854
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9803 - loss: 0.0637 - val_accuracy: 0.9857 - val_loss: 0.0588
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 52ms/step - accuracy: 0.9858 - loss: 0.0481 - val_accuracy: 0.9858 - val_loss: 0.0513


2026/11/19 01:25:22 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Experiment: CNN_Experiment_1
Test Accuracy: 0.9837
Test Loss: 0.0488
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 51ms/step - accuracy: 0.8974 - loss: 0.3638 - val_accuracy: 0.9723 - val_loss: 0.0991
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 42s 53ms/step - accuracy: 0.9716 - loss: 0.0937 - val_accuracy: 0.9758 - val_loss: 0.0855
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9806 - loss: 0.0655 - val_accuracy: 0.9855 - val_loss: 0.0505


2026/11/19 01:27:17 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Experiment: CNN_Experiment_2
Test Accuracy: 0.9843
Test Loss: 0.0490
Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 51ms/step - accuracy: 0.9288 - loss: 0.2466 - val_accuracy: 0.9787 - val_loss: 0.0679
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 42s 53ms/step - accuracy: 0.9804 - loss: 0.0631 - val_accuracy: 0.9857 - val_loss: 0.0474
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9868 - loss: 0.0432 - val_accuracy: 0.9892 - val_loss: 0.0394
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 52ms/step - accuracy: 0.9889 - loss: 0.0351 - val_accuracy: 0.9897 - val_loss: 0.0395
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 52ms/step - accuracy: 0.9916 - loss: 0.0278 - val_accuracy: 0.9893 - val_loss: 0.0345


2026/11/19 01:29:55 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Experiment: CNN_Experiment_3
Test Accuracy: 0.9888
Test Loss: 0.0318

All three experiments completed!


In [9]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

comparison = runs[
    [
        "tags.mlflow.runName",
        "params.learning_rate",
        "params.epochs",
        "metrics.test_accuracy",
        "metrics.test_loss"
    ]
]

comparison

,tags.mlflow.runName,params.learning_rate,params.epochs,metrics.test_accuracy,metrics.test_loss
0,CNN_Experiment_3,0.001,5,0.9888,0.031804
1,CNN_Experiment_2,0.0005,3,0.9843,0.048957
2,CNN_Experiment_1,0.001,3,0.9837,0.048849


In [10]:
best_run = runs.loc[runs["metrics.test_accuracy"].idxmax()]

best_run_id = best_run["run_id"]

print("Best Experiment:", best_run["tags.mlflow.runName"])
print("Best Test Accuracy:", best_run["metrics.test_accuracy"])
print("Best Run ID:", best_run_id)

Best Experiment: CNN_Experiment_3
Best Test Accuracy: 0.9887999892234802
Best Run ID: 932248a38ad24ced8c16b76eae9a88b6


In [14]:
client = mlflow.tracking.MlflowClient()

artifacts = client.list_artifacts(best_run_id)

for artifact in artifacts:
    print("Artifact:", artifact.path)
    print("Is directory:", artifact.is_dir)

In [15]:
# Inspect the best run's metadata

best_run_details = client.get_run(best_run_id)

print("Run ID:", best_run_details.info.run_id)
print("Artifact URI:", best_run_details.info.artifact_uri)
print("Status:", best_run_details.info.status)
print("Logged parameters:", best_run_details.data.params)
print("Logged metrics:", best_run_details.data.metrics)

Run ID: 932248a38ad24ced8c16b76eae9a88b6
Artifact URI: /content/mlruns/1/932248a38ad24ced8c16b76eae9a88b6/artifacts
Status: FINISHED
Logged parameters: {'learning_rate': '0.001', 'epochs': '5', 'batch_size': '128', 'optimizer': 'Adam', 'architecture': 'CNN'}
Logged metrics: {'test_loss': 0.03180370107293129, 'test_accuracy': 0.9887999892234802, 'final_train_accuracy': 0.991611123085022, 'final_val_accuracy': 0.9893333315849304}


In [17]:
print("Returned object type:", type(logged_models))
print("Returned data:", logged_models)

Returned object type: <class 'pandas.core.frame.DataFrame'>
Returned data:                                    artifact_location  creation_timestamp  \
0  /content/mlruns/1/models/m-2f7b467260f24cf5a3a...       1795051795918   
1  /content/mlruns/1/models/m-cc5dd302b4c44ddf8b2...       1795051637946   
2  /content/mlruns/1/models/m-0b7b550784d3445db5a...       1795051522392   

  experiment_id  last_updated_timestamp  \
0             1           1795051802351   
1             1           1795051644376   
2             1           1795051530730   

                                             metrics  \
0  [<Metric: dataset_digest=None, dataset_name=No...   
1  [<Metric: dataset_digest=None, dataset_name=No...   
2  [<Metric: dataset_digest=None, dataset_name=No...   

                             model_id model_type       name  \
0  m-2f7b467260f24cf5a3af5fa74a5b58a6       None  cnn_model   
1  m-cc5dd302b4c44ddf8b25d9f755f43a1e       None  cnn_model   
2  m-0b7b550784d3445db5a37e27c977

In [18]:
best_model_id = logged_models.loc[
    logged_models["source_run_id"] == best_run_id,
    "model_id"
].iloc[0]

model_uri = f"models:/{best_model_id}"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="MNIST_CNN_Best_Model"
)

print("Model registered successfully!")
print("Registered model:", registered_model.name)
print("Model version:", registered_model.version)

Model registered successfully!
Registered model: MNIST_CNN_Best_Model
Model version: 1


Successfully registered model 'MNIST_CNN_Best_Model'.
Created version '1' of model 'MNIST_CNN_Best_Model'.


In [19]:
client = mlflow.tracking.MlflowClient()

model_details = client.get_registered_model(
    "MNIST_CNN_Best_Model"
)

print("Registered Model:", model_details.name)

for version in model_details.latest_versions:
    print("Version:", version.version)
    print("Status:", version.status)
    print("Source:", version.source)
    print("Run ID:", version.run_id)

Registered Model: MNIST_CNN_Best_Model
Version: 1
Status: READY
Source: models:/m-2f7b467260f24cf5a3af5fa74a5b58a6
Run ID: 932248a38ad24ced8c16b76eae9a88b6


In [20]:
client.update_model_version(
    name="MNIST_CNN_Best_Model",
    version="1",
    description="CNN trained on the MNIST dataset using TensorFlow. "
                "Best experiment achieved 98.88% test accuracy "
                "with a test loss of 0.0318."
)

client.set_model_version_tag(
    name="MNIST_CNN_Best_Model",
    version="1",
    key="model_type",
    value="CNN"
)

client.set_model_version_tag(
    name="MNIST_CNN_Best_Model",
    version="1",
    key="dataset",
    value="MNIST"
)

client.set_model_version_tag(
    name="MNIST_CNN_Best_Model",
    version="1",
    key="experiment",
    value="CNN_Experiment_3"
)

print("Model description and tags updated successfully!")

Model description and tags updated successfully!


In [21]:
version_details = client.get_model_version(
    name="MNIST_CNN_Best_Model",
    version="1"
)

print("Model Name:", version_details.name)
print("Version:", version_details.version)
print("Status:", version_details.status)
print("Description:", version_details.description)
print("Tags:", version_details.tags)

Model Name: MNIST_CNN_Best_Model
Version: 1
Status: READY
Description: CNN trained on the MNIST dataset using TensorFlow. Best experiment achieved 98.88% test accuracy with a test loss of 0.0318.
Tags: {'model_type': 'CNN', 'dataset': 'MNIST', 'experiment': 'CNN_Experiment_3'}


In [22]:
loaded_model = mlflow.tensorflow.load_model(
    model_uri="models:/MNIST_CNN_Best_Model/1"
)

print("Registered model loaded successfully!")

Registered model loaded successfully!


In [23]:
sample_image = x_test[0:1]
prediction = loaded_model.predict(sample_image, verbose=0)

predicted_digit = np.argmax(prediction)
actual_digit = y_test[0]

print("Predicted Digit:", predicted_digit)
print("Actual Digit:", actual_digit)
print("Prediction Confidence:", round(float(np.max(prediction)) * 100, 2), "%")

Predicted Digit: 7
Actual Digit: 7
Prediction Confidence: 99.99 %
